# run_experiment

One experiment, end to end, from a settings file. Works on Colab and on a rented
GPU box; the only difference is which settings file you name.

Edit the next cell and nothing else. To run a second experiment, change
`SETTINGS_FILE` and re-run from **Run the experiment** downwards. The model stays
downloaded and the environment stays installed, so each extra run costs only its
own training time.

On a rented machine that expires, run the persist cell after every experiment
rather than at the end. Everything on that box is deleted when it shuts down.

In [ ]:
# Clone the pipeline branch when the code is not already here, then pin.
# run_me.py must run before the install: it re-pins requirements to whatever this
# machine already ships, which keeps its CUDA-matched torch instead of letting pip
# swap it out.
import os, subprocess, sys

REPO = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", "pipeline", REPO], check=True)
    os.chdir("Decision_Task_Database_Experiments")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, so they are easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
print("repo ready in", os.getcwd(), "- now run the install cell below")


In [ ]:
# %pip is the IPython magic that installs into THIS kernel's environment. A plain
# pip subprocess can land packages somewhere the kernel cannot import from, which
# surfaces later as ModuleNotFoundError on numpy or torch.
%pip install -q -r requirements.txt


In [ ]:
# Confirm the kernel can import what the pipeline needs. If anything is missing,
# restart the kernel (Kernel -> Restart Kernel) and run the install cell again.
import importlib.util, sys

print("kernel python:", sys.executable)
missing = [m for m in ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft")
           if importlib.util.find_spec(m) is None]
print("MISSING:", missing) if missing else print("all imports available")


## Set up the machine

In [ ]:
# What hardware is this? Harmless if there is no GPU.
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, use USE_TEST_CONFIG = True" 

In [ ]:
# Clone the pipeline branch when the code is not already here, then install.
# run_me.py must run before the install: it re-pins requirements to whatever this
# machine already ships, which keeps its CUDA-matched torch instead of letting pip
# swap it out.
import os, subprocess, sys

REPO = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", "pipeline", REPO], check=True)
    os.chdir("Decision_Task_Database_Experiments")
subprocess.run([sys.executable, "run_me.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Keep model downloads in one folder, so they are easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
print("ready in", os.getcwd())

In [ ]:
# Confirm the GPU matches what the settings assume. bf16 needs Ampere or newer:
# the h100 configs ask for it and the loader refuses rather than falling back,
# while the Colab configs ask for fp16, which a T4 can do.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
else:
    print("no CUDA device: only USE_TEST_CONFIG = True will run here")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or
# in a saved notebook. Needed only for the persist cells: the local backend and the
# public Qwen3 weights do not require a token.
import getpass, os

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face WRITE token (blank to skip): ")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi

    who = HfApi(token=os.environ["HF_TOKEN"]).whoami()
    owner = HUB_REPO.split("/")[0]
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Pushes will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Fix cell 1 to  HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\"  and re-run it.")
else:
    print("no token: the run will work, but persist will refuse and nothing survives the machine")


## Run the experiment

On a rented box, go in this order and persist after each one. The smoke config
exercises the CUDA path, the estimator, the report parser and the results writer
in a few minutes, which is the cheapest place to find a broken setting.

    configs/h100_smoke.json  ->  configs/h100_plunkett_8b.json  ->  configs/h100_plunkett_14b.json

In [ ]:
from src.config import load
from src.pipeline import run

cfg = load(None if USE_TEST_CONFIG else SETTINGS_FILE, use_test=USE_TEST_CONFIG)
rows = run(cfg)

## Save it off the machine

Results are small and worth pushing often. Adapters are tens of megabytes, so
pass `checkpoints=False` for a quick repeat push mid-run.

In [ ]:
from src.persist import push_run

push_run(rows[-1]["run_id"], HUB_REPO)

In [ ]:
# Before you walk away: sweep everything still on disk, including any run that
# was interrupted part way. The pipeline appends results as it goes, so a partial
# run is still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)